In [ ]:
# ============================================================
# BLUE EAGLE SECTOR ROTATION DASHBOARD
# Momentum + Analyst Recommendation Momentum
# Google Colab | One-cell version
# Direct WRDS SQLAlchemy connection
# ============================================================

# -----------------------------
# 0) Install packages
# -----------------------------
import sys, subprocess, pkgutil

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["psycopg2-binary", "sqlalchemy", "fredapi", "plotly", "ipywidgets"]
for pkg in required:
    mod = pkg.replace("-", "_")
    if pkgutil.find_loader(mod) is None:
        pip_install(pkg)

# -----------------------------
# 1) Imports
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from fredapi import Fred
from getpass import getpass
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from urllib.parse import quote_plus

# -----------------------------
# 2) Parameters
# -----------------------------
START_DATE = "2018-01-01"
END_DATE   = "2024-12-31"

DEFAULT_LOOKBACK_MONTHS = 6
DEFAULT_SKIP_MONTHS     = 1
DEFAULT_TOP_N           = 3
DEFAULT_BOTTOM_N        = 3

MOMENTUM_WEIGHT = 0.70
ANALYST_WEIGHT  = 0.30

SAVE_CSV_OUTPUTS = False

# -----------------------------
# 3) Helper functions
# -----------------------------
GICS_MAP = {
    "10": "Energy",
    "15": "Materials",
    "20": "Industrials",
    "25": "Consumer Discretionary",
    "30": "Consumer Staples",
    "35": "Health Care",
    "40": "Financials",
    "45": "Information Technology",
    "50": "Communication Services",
    "55": "Utilities",
    "60": "Real Estate"
}

def coerce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def annualized_return(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    cum = (1 + r).prod()
    years = len(r) / 12.0
    if years <= 0 or cum <= 0:
        return np.nan
    return cum ** (1 / years) - 1

def annualized_vol(r):
    r = pd.Series(r).dropna()
    if len(r) < 2:
        return np.nan
    return r.std() * np.sqrt(12)

def sharpe_ratio(r, rf=0.0):
    ar = annualized_return(r)
    av = annualized_vol(r)
    if pd.isna(ar) or pd.isna(av) or av == 0:
        return np.nan
    return (ar - rf) / av

def max_drawdown(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    wealth = (1 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1
    return dd.min()

def compute_compound_return(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return np.prod(1 + x) - 1

def traffic_light_from_rank(rank, n_sectors, top_n=3, bottom_n=3):
    if pd.isna(rank):
        return np.nan
    if rank <= top_n:
        return "Green"
    elif rank > n_sectors - bottom_n:
        return "Red"
    else:
        return "Yellow"

def zscore_by_date(df, value_col, new_col):
    out = df.copy()
    def _z(x):
        sd = x.std(ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(np.nan, index=x.index)
        return (x - x.mean()) / sd
    out[new_col] = out.groupby("date")[value_col].transform(_z)
    return out

def safe_hit_rate(strategy, benchmark):
    s = pd.Series(strategy)
    b = pd.Series(benchmark)
    mask = s.notna() & b.notna()
    if mask.sum() == 0:
        return np.nan
    return (s[mask] > b[mask]).mean()

# -----------------------------
# 4) WRDS connection helpers
# -----------------------------
print("Enter your WRDS credentials.")
wrds_username = input("WRDS username: ").strip()
wrds_password = getpass("WRDS password: ").strip()

WRDS_HOST = "wrds-pgdata.wharton.upenn.edu"
WRDS_PORT = 9737
WRDS_DB   = "wrds"

engine = None

def connect_wrds_engine(max_tries=3, wait_seconds=5):
    global engine
    for attempt in range(1, max_tries + 1):
        try:
            password_escaped = quote_plus(wrds_password)
            conn_str = (
                f"postgresql+psycopg2://{wrds_username}:{password_escaped}"
                f"@{WRDS_HOST}:{WRDS_PORT}/{WRDS_DB}"
            )
            engine = create_engine(
                conn_str,
                connect_args={
                    "sslmode": "require",
                    "connect_timeout": 30,
                    "application_name": "colab_sector_rotation_multifactor",
                    "keepalives": 1,
                    "keepalives_idle": 30,
                    "keepalives_interval": 10,
                    "keepalives_count": 5,
                },
                pool_pre_ping=True,
                pool_recycle=180,
            )
            with engine.connect() as conn:
                conn.exec_driver_sql("select 1")
            print("WRDS connection successful.")
            return engine
        except Exception as e:
            print(f"WRDS connection failed on attempt {attempt}: {e}")
            if attempt < max_tries:
                time.sleep(wait_seconds)
            else:
                raise

def run_wrds_sql(sql, date_cols=None, max_tries=3, wait_seconds=5):
    global engine
    for attempt in range(1, max_tries + 1):
        try:
            return pd.read_sql_query(sql, engine, parse_dates=date_cols)
        except Exception as e:
            print(f"Query failed on attempt {attempt}: {e}")
            if attempt < max_tries:
                print("Reconnecting to WRDS...")
                try:
                    engine.dispose()
                except:
                    pass
                time.sleep(wait_seconds)
                connect_wrds_engine()
            else:
                raise

connect_wrds_engine()

# -----------------------------
# 5) Pull CRSP monthly stock data
# -----------------------------
print("Pulling CRSP monthly data...")

crsp_sql = f"""
select
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    b.shrcd,
    b.exchcd
from crsp.msf as a
join crsp.msenames as b
    on a.permno = b.permno
   and b.namedt <= a.date
   and a.date <= b.nameendt
where a.date between '{START_DATE}' and '{END_DATE}'
  and b.shrcd in (10, 11)
  and b.exchcd in (1, 2, 3)
"""

crsp = run_wrds_sql(crsp_sql, date_cols=["date"])
crsp = coerce_numeric(crsp, ["ret", "prc", "shrout", "shrcd", "exchcd"])
crsp = crsp.dropna(subset=["date", "permno", "ret", "prc", "shrout"]).copy()
crsp["mkt_cap"] = crsp["prc"].abs() * crsp["shrout"]
crsp["date"] = pd.to_datetime(crsp["date"]) + pd.offsets.MonthEnd(0)

print(f"CRSP rows: {len(crsp):,}")

# -----------------------------
# 6) Pull CCM link table
# -----------------------------
print("Pulling CRSP-Compustat link table...")

ccm_sql = """
select
    gvkey,
    lpermno as permno,
    linktype,
    linkprim,
    linkdt,
    linkenddt
from crsp.ccmxpf_linktable
where lpermno is not null
  and linktype in ('LU','LC','LS')
  and linkprim in ('P','C')
"""
ccm = run_wrds_sql(ccm_sql, date_cols=["linkdt", "linkenddt"])
ccm["linkenddt"] = ccm["linkenddt"].fillna(pd.Timestamp("2100-12-31"))

# -----------------------------
# 7) Pull Compustat GICS sector
# -----------------------------
print("Pulling Compustat GICS sectors...")

comp_sql = """
select
    gvkey,
    conm,
    gsector
from comp.company
where gsector is not null
"""
comp = run_wrds_sql(comp_sql)
comp["gsector"] = comp["gsector"].astype(str).str.strip()

# -----------------------------
# 8) Build stock-level base panel
# -----------------------------
print("Building stock-level base panel...")

stocks = crsp.merge(ccm, how="left", on="permno")
stocks = stocks[
    (stocks["date"] >= stocks["linkdt"]) &
    (stocks["date"] <= stocks["linkenddt"])
].copy()

stocks = stocks.sort_values(["permno", "date", "gvkey"]).drop_duplicates(["permno", "date"], keep="first")
stocks = stocks.merge(comp[["gvkey", "gsector"]], how="left", on="gvkey")

stocks["gsector"] = stocks["gsector"].astype(str).str.strip()
stocks["sector"] = stocks["gsector"].map(GICS_MAP)
stocks = stocks[stocks["sector"].notna()].copy()

stocks = stocks.sort_values(["permno", "date"]).copy()
stocks["lag_mkt_cap"] = stocks.groupby("permno")["mkt_cap"].shift(1)
stocks = stocks.dropna(subset=["lag_mkt_cap", "ret"]).copy()
stocks = stocks[stocks["lag_mkt_cap"] > 0].copy()

print(f"Stock panel rows: {len(stocks):,}")

# -----------------------------
# 9) Build sector returns
# -----------------------------
print("Building value-weighted sector returns...")

sector_totals = stocks.groupby(["date", "sector"], as_index=False)["lag_mkt_cap"].sum()
sector_totals = sector_totals.rename(columns={"lag_mkt_cap": "sector_lag_cap"})

stocks = stocks.merge(sector_totals, on=["date", "sector"], how="left")
stocks["weight"] = stocks["lag_mkt_cap"] / stocks["sector_lag_cap"]

sector_returns = (
    stocks.groupby(["date", "sector"], as_index=False)
    .apply(lambda x: pd.Series({
        "sector_ret": np.sum(x["weight"] * x["ret"]),
        "n_stocks": x["permno"].nunique(),
        "sector_lag_cap": x["sector_lag_cap"].iloc[0]
    }))
    .reset_index(drop=True)
)

all_dates = pd.date_range(sector_returns["date"].min(), sector_returns["date"].max(), freq="M")
all_sectors = list(GICS_MAP.values())
grid = pd.MultiIndex.from_product([all_dates, all_sectors], names=["date", "sector"]).to_frame(index=False)

sector_returns = grid.merge(sector_returns, how="left", on=["date", "sector"]).sort_values(["sector", "date"])
print(f"Sector return rows: {len(sector_returns):,}")

# -----------------------------
# 10) Discover IBES recommendation table
# -----------------------------
print("Discovering IBES recommendation table...")

table_candidates = [
    ("ibes", "recdsum"),
    ("tr_ibes", "recdsum"),
    ("ibes", "recdsumus"),
    ("tr_ibes", "recdsumus"),
]

ticker_col_candidates = ["ticker", "ibtic", "oftic"]
date_col_candidates   = ["statpers", "date", "anndats"]
rec_col_candidates    = ["meanrec", "meanrecom", "mean_rec"]
nanalyst_candidates   = ["numest", "numrec", "num_analysts"]

def get_table_columns(schema_name, table_name):
    sql = f"""
    select column_name
    from information_schema.columns
    where table_schema = '{schema_name}'
      and table_name = '{table_name}'
    """
    df = run_wrds_sql(sql)
    return [c.lower() for c in df["column_name"].tolist()] if len(df) > 0 else []

def first_match(candidates, available):
    for c in candidates:
        if c in available:
            return c
    return None

rec_table_schema = None
rec_table_name   = None
rec_ticker_col   = None
rec_date_col     = None
rec_value_col    = None
rec_numest_col   = None

for schema_name, table_name in table_candidates:
    cols = get_table_columns(schema_name, table_name)
    if len(cols) == 0:
        continue
    tcol = first_match(ticker_col_candidates, cols)
    dcol = first_match(date_col_candidates, cols)
    rcol = first_match(rec_col_candidates, cols)
    ncol = first_match(nanalyst_candidates, cols)
    if tcol and dcol and rcol:
        rec_table_schema = schema_name
        rec_table_name   = table_name
        rec_ticker_col   = tcol
        rec_date_col     = dcol
        rec_value_col    = rcol
        rec_numest_col   = ncol
        break

if rec_table_schema is None:
    raise ValueError(
        "Could not find an IBES recommendation summary table automatically. "
        "Run a WRDS table search for recommendation summary data and update table_candidates."
    )

print(f"Using recommendation table: {rec_table_schema}.{rec_table_name}")
print(f"Ticker col: {rec_ticker_col} | Date col: {rec_date_col} | Rec col: {rec_value_col} | Num analysts col: {rec_numest_col}")

# -----------------------------
# 11) Pull recommendation data
# -----------------------------
print("Pulling analyst recommendation data...")

rec_start = (pd.to_datetime(START_DATE) - pd.DateOffset(months=18)).strftime("%Y-%m-%d")

extra_cols = f", {rec_numest_col}" if rec_numest_col is not None else ""

rec_sql = f"""
select
    {rec_ticker_col} as ibes_ticker,
    {rec_date_col}   as stat_date,
    {rec_value_col}  as mean_rec
    {extra_cols}
from {rec_table_schema}.{rec_table_name}
where {rec_date_col} between '{rec_start}' and '{END_DATE}'
"""

rec = run_wrds_sql(rec_sql, date_cols=["stat_date"])
rec["ibes_ticker"] = rec["ibes_ticker"].astype(str).str.strip()
rec = coerce_numeric(rec, ["mean_rec"] + ([rec_numest_col] if rec_numest_col is not None else []))
rec = rec.dropna(subset=["ibes_ticker", "stat_date", "mean_rec"]).copy()

if rec_numest_col is not None:
    rec = rec[rec[rec_numest_col].fillna(0) >= 1].copy()

rec["date"] = pd.to_datetime(rec["stat_date"]) + pd.offsets.MonthEnd(0)

# keep the latest recommendation snapshot in each month per ticker
rec = rec.sort_values(["ibes_ticker", "stat_date"]).drop_duplicates(["ibes_ticker", "date"], keep="last")
print(f"Recommendation rows after monthly snapshot logic: {len(rec):,}")

# -----------------------------
# 12) Discover CRSP-IBES linking table
# -----------------------------
print("Discovering IBES-CRSP link table...")

link_candidates = [
    ("wrdsapps", "ibcrsphist"),
    ("ibes", "iclink"),
    ("tr_ibes", "iclink"),
]

link_ticker_candidates = ["ticker", "ibtic"]
link_permno_candidates = ["permno", "lpermno"]
link_start_candidates  = ["sdate", "fdate", "namedt", "linkdt"]
link_end_candidates    = ["edate", "ldate", "nameendt", "linkenddt"]
link_score_candidates  = ["score"]

link_schema = None
link_table  = None
link_ticker_col = None
link_permno_col = None
link_start_col  = None
link_end_col    = None
link_score_col  = None

for schema_name, table_name in link_candidates:
    cols = get_table_columns(schema_name, table_name)
    if len(cols) == 0:
        continue
    tcol = first_match(link_ticker_candidates, cols)
    pcol = first_match(link_permno_candidates, cols)
    scol = first_match(link_start_candidates, cols)
    ecol = first_match(link_end_candidates, cols)
    qcol = first_match(link_score_candidates, cols)
    if tcol and pcol and scol and ecol:
        link_schema = schema_name
        link_table  = table_name
        link_ticker_col = tcol
        link_permno_col = pcol
        link_start_col  = scol
        link_end_col    = ecol
        link_score_col  = qcol
        break

if link_schema is None:
    raise ValueError(
        "Could not find an IBES-CRSP linking table automatically. "
        "Update link_candidates for your WRDS account."
    )

print(f"Using link table: {link_schema}.{link_table}")

extra_score = f", {link_score_col}" if link_score_col is not None else ""

link_sql = f"""
select
    {link_ticker_col} as ibes_ticker,
    {link_permno_col} as permno,
    {link_start_col}  as link_start,
    {link_end_col}    as link_end
    {extra_score}
from {link_schema}.{link_table}
where {link_ticker_col} is not null
  and {link_permno_col} is not null
"""

link = run_wrds_sql(link_sql, date_cols=["link_start", "link_end"])
link["link_end"] = link["link_end"].fillna(pd.Timestamp("2100-12-31"))
link["ibes_ticker"] = link["ibes_ticker"].astype(str).str.strip()

# -----------------------------
# 13) Map recommendations to permno
# -----------------------------
print("Mapping recommendations to CRSP permnos...")

rec_linked = rec.merge(link, how="left", on="ibes_ticker")
rec_linked = rec_linked[
    (rec_linked["date"] >= rec_linked["link_start"]) &
    (rec_linked["date"] <= rec_linked["link_end"])
].copy()

if link_score_col is not None and link_score_col in rec_linked.columns:
    rec_linked = rec_linked.sort_values(["ibes_ticker", "date", link_score_col]).drop_duplicates(
        ["ibes_ticker", "date"], keep="first"
    )
else:
    rec_linked = rec_linked.sort_values(["ibes_ticker", "date", "permno"]).drop_duplicates(
        ["ibes_ticker", "date"], keep="first"
    )

rec_linked = rec_linked.dropna(subset=["permno"]).copy()
rec_linked["permno"] = pd.to_numeric(rec_linked["permno"], errors="coerce")
rec_linked = rec_linked.dropna(subset=["permno"]).copy()
rec_linked["permno"] = rec_linked["permno"].astype(int)

print(f"Linked recommendation rows: {len(rec_linked):,}")

# -----------------------------
# 14) Build analyst recommendation momentum factor
# -----------------------------
print("Building analyst recommendation momentum factor...")

# lower recommendation score is better, so:
# improvement = recommendation_12m_ago - recommendation_current
# positive number = sentiment improved

rec_linked = rec_linked.sort_values(["permno", "date"]).copy()
rec_linked["mean_rec_lag_12m"] = rec_linked.groupby("permno")["mean_rec"].shift(12)
rec_linked["analyst_change_12m"] = rec_linked["mean_rec_lag_12m"] - rec_linked["mean_rec"]

analyst_stock = rec_linked[["permno", "date", "mean_rec", "mean_rec_lag_12m", "analyst_change_12m"]].copy()

stocks_plus = stocks.merge(
    analyst_stock,
    how="left",
    on=["permno", "date"]
)

# market-cap-weighted sector analyst score
analyst_sector = (
    stocks_plus.dropna(subset=["analyst_change_12m"])
    .groupby(["date", "sector"], as_index=False)
    .apply(lambda x: pd.Series({
        "analyst_raw": np.sum(x["lag_mkt_cap"] * x["analyst_change_12m"]) / np.sum(x["lag_mkt_cap"]),
        "analyst_names": x["permno"].nunique()
    }))
    .reset_index(drop=True)
)

sector_factors = sector_returns.merge(analyst_sector, how="left", on=["date", "sector"]).sort_values(["sector", "date"])

# -----------------------------
# 15) Signal engine
# -----------------------------
def build_signals(sector_df, lookback_months=6, skip_months=1, top_n=3, bottom_n=3):
    df = sector_df.copy().sort_values(["sector", "date"])

    # momentum factor
    df["momentum_raw"] = (
        df.groupby("sector")["sector_ret"]
        .transform(lambda x: x.shift(skip_months).rolling(lookback_months).apply(compute_compound_return, raw=False))
    )

    # z-score each component cross-sectionally
    df = zscore_by_date(df, "momentum_raw", "momentum_z")
    df = zscore_by_date(df, "analyst_raw",  "analyst_z")

    # if analyst score missing, treat as neutral instead of dropping entire sector
    df["momentum_z_filled"] = df["momentum_z"].fillna(0)
    df["analyst_z_filled"]  = df["analyst_z"].fillna(0)

    # final combined score
    df["final_score"] = (
        MOMENTUM_WEIGHT * df["momentum_z_filled"] +
        ANALYST_WEIGHT  * df["analyst_z_filled"]
    )

    df["rank"] = df.groupby("date")["final_score"].rank(ascending=False, method="first")
    df["n_sectors"] = df.groupby("date")["sector"].transform("count")

    df["signal"] = df.apply(
        lambda r: traffic_light_from_rank(
            r["rank"],
            int(r["n_sectors"]) if not pd.isna(r["n_sectors"]) else 11,
            top_n,
            bottom_n
        ),
        axis=1
    )

    df["fwd_1m_ret"] = df.groupby("sector")["sector_ret"].shift(-1)
    return df

# -----------------------------
# 16) Backtest engine
# -----------------------------
def run_backtest(signal_df, top_n=3):
    df = signal_df.copy()
    valid = df.dropna(subset=["rank", "fwd_1m_ret"]).copy()

    valid["selected"] = valid["rank"] <= top_n

    port = valid[valid["selected"]].groupby("date", as_index=False).agg(
        strategy_ret=("fwd_1m_ret", "mean")
    )

    bench = valid.groupby("date", as_index=False).agg(
        benchmark_ret=("fwd_1m_ret", "mean")
    )

    green = valid[valid["signal"] == "Green"].groupby("date", as_index=False).agg(
        green_ret=("fwd_1m_ret", "mean")
    )

    red = valid[valid["signal"] == "Red"].groupby("date", as_index=False).agg(
        red_ret=("fwd_1m_ret", "mean")
    )

    out = (
        port.merge(bench, on="date", how="outer")
            .merge(green, on="date", how="left")
            .merge(red, on="date", how="left")
            .sort_values("date")
            .reset_index(drop=True)
    )

    out["strategy_cum"] = (1 + out["strategy_ret"].fillna(0)).cumprod()
    out["benchmark_cum"] = (1 + out["benchmark_ret"].fillna(0)).cumprod()
    out["spread_ret"] = out["green_ret"] - out["red_ret"]
    out["spread_cum"] = (1 + out["spread_ret"].fillna(0)).cumprod()

    return out

def summarize_performance(bt):
    return pd.DataFrame({
        "Metric": [
            "Annualized Return",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Max Drawdown",
            "Hit Rate vs Benchmark"
        ],
        "Strategy": [
            annualized_return(bt["strategy_ret"]),
            annualized_vol(bt["strategy_ret"]),
            sharpe_ratio(bt["strategy_ret"]),
            max_drawdown(bt["strategy_ret"]),
            safe_hit_rate(bt["strategy_ret"], bt["benchmark_ret"])
        ],
        "Benchmark": [
            annualized_return(bt["benchmark_ret"]),
            annualized_vol(bt["benchmark_ret"]),
            sharpe_ratio(bt["benchmark_ret"]),
            max_drawdown(bt["benchmark_ret"]),
            np.nan
        ],
        "Green-Red Spread": [
            annualized_return(bt["spread_ret"]),
            annualized_vol(bt["spread_ret"]),
            sharpe_ratio(bt["spread_ret"]),
            max_drawdown(bt["spread_ret"]),
            np.nan
        ]
    })

def display_perf_table(perf):
    perf_fmt = perf.copy()
    pct_metrics = {"Annualized Return", "Annualized Volatility", "Max Drawdown", "Hit Rate vs Benchmark"}
    for col in ["Strategy", "Benchmark", "Green-Red Spread"]:
        perf_fmt[col] = perf_fmt.apply(
            lambda row: f"{row[col]:.2%}" if (row["Metric"] in pct_metrics and pd.notna(row[col]))
            else (f"{row[col]:.2f}" if pd.notna(row[col]) else ""),
            axis=1
        )
    display(perf_fmt)

# -----------------------------
# 17) Initial build
# -----------------------------
signals = build_signals(
    sector_factors,
    lookback_months=DEFAULT_LOOKBACK_MONTHS,
    skip_months=DEFAULT_SKIP_MONTHS,
    top_n=DEFAULT_TOP_N,
    bottom_n=DEFAULT_BOTTOM_N
)

backtest = run_backtest(signals, top_n=DEFAULT_TOP_N)

# -----------------------------
# 18) Dashboard renderer
# -----------------------------
def render_dashboard(start_date, end_date, lookback_months, skip_months, top_n, bottom_n):
    clear_output(wait=True)

    sig = build_signals(
        sector_factors,
        lookback_months=int(lookback_months),
        skip_months=int(skip_months),
        top_n=int(top_n),
        bottom_n=int(bottom_n)
    )

    bt = run_backtest(sig, top_n=int(top_n))

    start_date = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end_date = pd.to_datetime(end_date) + pd.offsets.MonthEnd(0)

    sig_f = sig[(sig["date"] >= start_date) & (sig["date"] <= end_date)].copy()
    bt_f = bt[(bt["date"] >= start_date) & (bt["date"] <= end_date)].copy()

    if len(sig_f) == 0 or len(bt_f) == 0:
        print("No data in selected range.")
        return

    current_date = sig_f["date"].max()
    current = sig_f[sig_f["date"] == current_date].sort_values("rank").copy()
    perf = summarize_performance(bt_f)

    bt_f["strategy_cum"] = (1 + bt_f["strategy_ret"].fillna(0)).cumprod()
    bt_f["benchmark_cum"] = (1 + bt_f["benchmark_ret"].fillna(0)).cumprod()
    bt_f["spread_cum"] = (1 + bt_f["spread_ret"].fillna(0)).cumprod()

    print("=" * 88)
    print("SECTOR ROTATION DASHBOARD: MOMENTUM + ANALYST RECOMMENDATION MOMENTUM")
    print("=" * 88)
    print(f"Date range: {start_date.date()} to {end_date.date()}")
    print(f"Lookback: {lookback_months} | Skip: {skip_months} | Top N: {top_n} | Bottom N: {bottom_n}")
    print(f"Factor weights: Momentum = {MOMENTUM_WEIGHT:.0%}, Analyst = {ANALYST_WEIGHT:.0%}")
    print(f"Latest signal date: {current_date.date()}")
    print("=" * 88)

    print("\nCURRENT MONTH SIGNAL TABLE")
    show_cols = [
        "date", "sector", "momentum_raw", "momentum_z",
        "analyst_raw", "analyst_z", "final_score",
        "rank", "signal", "n_stocks", "analyst_names"
    ]
    current_show = current[show_cols].copy()
    for c in ["momentum_raw", "momentum_z", "analyst_raw", "analyst_z", "final_score"]:
        if c in current_show.columns:
            current_show[c] = current_show[c].round(4)
    display(current_show.reset_index(drop=True))

    print("\nBACKTEST SUMMARY")
    display_perf_table(perf)

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["strategy_cum"], mode="lines", name="Strategy"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["benchmark_cum"], mode="lines", name="Benchmark"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["spread_cum"], mode="lines", name="Green-Red Spread"))
    fig1.update_layout(
        title="Cumulative Performance",
        xaxis_title="Date",
        yaxis_title="Growth of $1",
        template="plotly_white",
        width=1000,
        height=500
    )
    fig1.show()

    signal_to_num = {"Red": -1, "Yellow": 0, "Green": 1}
    hm = sig_f[["date", "sector", "signal"]].copy()
    hm["signal_num"] = hm["signal"].map(signal_to_num)
    heat = hm.pivot(index="sector", columns="date", values="signal_num").reindex(all_sectors)

    fig2 = px.imshow(
        heat,
        aspect="auto",
        title="Traffic Light Heatmap",
        color_continuous_scale=["red", "yellow", "green"],
        zmin=-1,
        zmax=1
    )
    fig2.update_layout(width=1200, height=500)
    fig2.show()

    fig3 = px.bar(
        current.sort_values("rank"),
        x="sector",
        y="final_score",
        color="signal",
        title=f"Current Combined Sector Score ({current_date.date()})",
        hover_data=["momentum_raw", "analyst_raw", "rank"]
    )
    fig3.update_layout(template="plotly_white", width=1000, height=500)
    fig3.show()

    fig4 = px.bar(
        current.sort_values("rank"),
        x="sector",
        y=["momentum_z", "analyst_z"],
        barmode="group",
        title=f"Factor Decomposition ({current_date.date()})"
    )
    fig4.update_layout(template="plotly_white", width=1100, height=500)
    fig4.show()

    if SAVE_CSV_OUTPUTS:
        sig_f.to_csv("multifactor_sector_signals.csv", index=False)
        bt_f.to_csv("multifactor_sector_backtest.csv", index=False)
        current_show.to_csv("multifactor_current_signal_table.csv", index=False)
        print("\nCSV files saved to current Colab working directory.")

# -----------------------------
# 19) Widgets
# -----------------------------
available_dates = pd.Series(sorted(sector_factors["date"].dropna().unique()))
min_date = available_dates.min().date()
max_date = available_dates.max().date()

display(HTML("<h2>Sector Rotation Model Dashboard: Momentum + Analyst Recommendation Momentum</h2>"))

start_picker = widgets.DatePicker(description="Start", value=min_date)
end_picker = widgets.DatePicker(description="End", value=max_date)

lookback_slider = widgets.IntSlider(value=DEFAULT_LOOKBACK_MONTHS, min=3, max=12, step=1, description="Lookback")
skip_slider = widgets.IntSlider(value=DEFAULT_SKIP_MONTHS, min=0, max=2, step=1, description="Skip")
topn_slider = widgets.IntSlider(value=DEFAULT_TOP_N, min=1, max=5, step=1, description="Top N")
bottomn_slider = widgets.IntSlider(value=DEFAULT_BOTTOM_N, min=1, max=5, step=1, description="Bottom N")
run_button = widgets.Button(description="Run Dashboard", button_style="success")
output = widgets.Output()

def on_run_clicked(_):
    with output:
        render_dashboard(
            start_date=start_picker.value,
            end_date=end_picker.value,
            lookback_months=lookback_slider.value,
            skip_months=skip_slider.value,
            top_n=topn_slider.value,
            bottom_n=bottomn_slider.value
        )

run_button.on_click(on_run_clicked)

controls = widgets.VBox([
    widgets.HBox([start_picker, end_picker]),
    widgets.HBox([lookback_slider, skip_slider]),
    widgets.HBox([topn_slider, bottomn_slider]),
    widgets.HBox([run_button]),
])

display(controls, output)

with output:
    render_dashboard(
        start_date=start_picker.value,
        end_date=end_picker.value,
        lookback_months=lookback_slider.value,
        skip_months=skip_slider.value,
        top_n=topn_slider.value,
        bottom_n=bottomn_slider.value
    )\

In [ ]:
# ============================================================
# BLUE EAGLE SECTOR ROTATION DASHBOARD
# Colab version using direct psycopg2 connection to WRDS
# This is more stable than wrds.Connection() in Colab
# ============================================================

# -----------------------------
# 0) Install packages
# -----------------------------
import sys, subprocess, pkgutil

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["psycopg2-binary", "sqlalchemy", "fredapi", "plotly", "ipywidgets"]
for pkg in required:
    if pkgutil.find_loader(pkg.replace("-", "_")) is None:
        pip_install(pkg)

# -----------------------------
# 1) Imports
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import psycopg2
from sqlalchemy import create_engine
from fredapi import Fred
from getpass import getpass
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from urllib.parse import quote_plus

# -----------------------------
# 2) User parameters
# -----------------------------
START_DATE = "2018-01-01"
END_DATE   = "2024-12-31"

DEFAULT_LOOKBACK_MONTHS = 6
DEFAULT_SKIP_MONTHS     = 1
DEFAULT_TOP_N           = 3
DEFAULT_BOTTOM_N        = 3

USE_MACRO_OVERLAY = False
SAVE_CSV_OUTPUTS  = False

# -----------------------------
# 3) Helper functions
# -----------------------------
GICS_MAP = {
    "10": "Energy",
    "15": "Materials",
    "20": "Industrials",
    "25": "Consumer Discretionary",
    "30": "Consumer Staples",
    "35": "Health Care",
    "40": "Financials",
    "45": "Information Technology",
    "50": "Communication Services",
    "55": "Utilities",
    "60": "Real Estate"
}

CYCLICALS = {
    "Energy", "Materials", "Industrials", "Consumer Discretionary",
    "Financials", "Information Technology", "Communication Services", "Real Estate"
}
DEFENSIVES = {"Consumer Staples", "Health Care", "Utilities"}

def coerce_numeric(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def annualized_return(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    cum = (1 + r).prod()
    years = len(r) / 12.0
    if years <= 0 or cum <= 0:
        return np.nan
    return cum ** (1 / years) - 1

def annualized_vol(r):
    r = pd.Series(r).dropna()
    if len(r) < 2:
        return np.nan
    return r.std() * np.sqrt(12)

def sharpe_ratio(r, rf=0.0):
    ar = annualized_return(r)
    av = annualized_vol(r)
    if pd.isna(ar) or pd.isna(av) or av == 0:
        return np.nan
    return (ar - rf) / av

def max_drawdown(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    wealth = (1 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1
    return dd.min()

def compute_compound_return(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return np.prod(1 + x) - 1

def traffic_light_from_rank(rank, n_sectors, top_n=3, bottom_n=3):
    if pd.isna(rank):
        return np.nan
    if rank <= top_n:
        return "Green"
    elif rank > n_sectors - bottom_n:
        return "Red"
    else:
        return "Yellow"

def zscore_by_date(df, value_col):
    out = df.copy()
    def _z(x):
        sd = x.std(ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(np.nan, index=x.index)
        return (x - x.mean()) / sd
    out["zscore"] = out.groupby("date")[value_col].transform(_z)
    return out

def safe_hit_rate(strategy, benchmark):
    s = pd.Series(strategy)
    b = pd.Series(benchmark)
    mask = s.notna() & b.notna()
    if mask.sum() == 0:
        return np.nan
    return (s[mask] > b[mask]).mean()

# -----------------------------
# 4) Direct WRDS connection
# -----------------------------
print("Enter your WRDS credentials.")
wrds_username = input("WRDS username: ").strip()
wrds_password = getpass("WRDS password: ").strip()

WRDS_HOST = "wrds-pgdata.wharton.upenn.edu"
WRDS_PORT = 9737
WRDS_DB   = "wrds"

engine = None

def connect_wrds_engine(max_tries=3, wait_seconds=5):
    global engine
    for attempt in range(1, max_tries + 1):
        try:
            password_escaped = quote_plus(wrds_password)
            conn_str = (
                f"postgresql+psycopg2://{wrds_username}:{password_escaped}"
                f"@{WRDS_HOST}:{WRDS_PORT}/{WRDS_DB}"
            )
            engine = create_engine(
                conn_str,
                connect_args={
                    "sslmode": "require",
                    "connect_timeout": 30,
                    "application_name": "colab_sector_rotation",
                    "keepalives": 1,
                    "keepalives_idle": 30,
                    "keepalives_interval": 10,
                    "keepalives_count": 5,
                },
                pool_pre_ping=True,
                pool_recycle=180,
            )
            # test connection
            with engine.connect() as conn:
                conn.exec_driver_sql("select 1")
            print("WRDS connection successful.")
            return engine
        except Exception as e:
            print(f"WRDS connection failed on attempt {attempt}: {e}")
            if attempt < max_tries:
                time.sleep(wait_seconds)
            else:
                raise

def run_wrds_sql(sql, date_cols=None, max_tries=3, wait_seconds=5):
    global engine
    for attempt in range(1, max_tries + 1):
        try:
            return pd.read_sql_query(sql, engine, parse_dates=date_cols)
        except Exception as e:
            print(f"Query failed on attempt {attempt}: {e}")
            if attempt < max_tries:
                print("Reconnecting to WRDS...")
                try:
                    engine.dispose()
                except:
                    pass
                time.sleep(wait_seconds)
                connect_wrds_engine()
            else:
                raise

connect_wrds_engine()

# -----------------------------
# 5) Pull CRSP monthly stock data
# -----------------------------
print("Pulling CRSP monthly data...")

crsp_sql = f"""
select
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    b.shrcd,
    b.exchcd
from crsp.msf as a
join crsp.msenames as b
    on a.permno = b.permno
   and b.namedt <= a.date
   and a.date <= b.nameendt
where a.date between '{START_DATE}' and '{END_DATE}'
  and b.shrcd in (10, 11)
  and b.exchcd in (1, 2, 3)
"""

crsp = run_wrds_sql(crsp_sql, date_cols=["date"])
crsp = coerce_numeric(crsp, ["ret", "prc", "shrout", "shrcd", "exchcd"])
crsp = crsp.dropna(subset=["date", "permno", "ret", "prc", "shrout"]).copy()
crsp["mkt_cap"] = crsp["prc"].abs() * crsp["shrout"]
crsp["date"] = pd.to_datetime(crsp["date"]) + pd.offsets.MonthEnd(0)

print(f"CRSP rows: {len(crsp):,}")

# -----------------------------
# 6) Pull CCM link table
# -----------------------------
print("Pulling CRSP-Compustat link table...")

ccm_sql = """
select
    gvkey,
    lpermno as permno,
    linktype,
    linkprim,
    linkdt,
    linkenddt
from crsp.ccmxpf_linktable
where lpermno is not null
  and linktype in ('LU','LC','LS')
  and linkprim in ('P','C')
"""
ccm = run_wrds_sql(ccm_sql, date_cols=["linkdt", "linkenddt"])
ccm["linkenddt"] = ccm["linkenddt"].fillna(pd.Timestamp("2100-12-31"))

# -----------------------------
# 7) Pull Compustat GICS sector
# -----------------------------
print("Pulling Compustat GICS sectors...")

comp_sql = """
select
    gvkey,
    conm,
    gsector
from comp.company
where gsector is not null
"""
comp = run_wrds_sql(comp_sql)
comp["gsector"] = comp["gsector"].astype(str).str.strip()

# -----------------------------
# 8) Merge CRSP -> CCM -> Compustat
# -----------------------------
print("Merging CRSP, CCM, and Compustat...")

merged = crsp.merge(ccm, how="left", on="permno")
merged = merged[
    (merged["date"] >= merged["linkdt"]) &
    (merged["date"] <= merged["linkenddt"])
].copy()

merged = merged.sort_values(["permno", "date", "gvkey"]).drop_duplicates(["permno", "date"], keep="first")
merged = merged.merge(comp[["gvkey", "gsector"]], how="left", on="gvkey")

merged["gsector"] = merged["gsector"].astype(str).str.strip()
merged["sector"] = merged["gsector"].map(GICS_MAP)
merged = merged[merged["sector"].notna()].copy()

print(f"Merged mapped rows: {len(merged):,}")

# -----------------------------
# 9) Build value-weighted sector returns
# -----------------------------
print("Building value-weighted sector returns...")

merged = merged.sort_values(["permno", "date"]).copy()
merged["lag_mkt_cap"] = merged.groupby("permno")["mkt_cap"].shift(1)
merged = merged.dropna(subset=["lag_mkt_cap", "ret"]).copy()
merged = merged[merged["lag_mkt_cap"] > 0].copy()

sector_totals = merged.groupby(["date", "sector"], as_index=False)["lag_mkt_cap"].sum()
sector_totals = sector_totals.rename(columns={"lag_mkt_cap": "sector_lag_cap"})

merged = merged.merge(sector_totals, on=["date", "sector"], how="left")
merged["weight"] = merged["lag_mkt_cap"] / merged["sector_lag_cap"]

sector_returns = (
    merged.groupby(["date", "sector"], as_index=False)
    .apply(lambda x: pd.Series({
        "sector_ret": np.sum(x["weight"] * x["ret"]),
        "n_stocks": x["permno"].nunique(),
        "sector_lag_cap": x["sector_lag_cap"].iloc[0]
    }))
    .reset_index(drop=True)
)

all_dates = pd.date_range(sector_returns["date"].min(), sector_returns["date"].max(), freq="M")
all_sectors = list(GICS_MAP.values())
grid = pd.MultiIndex.from_product([all_dates, all_sectors], names=["date", "sector"]).to_frame(index=False)

sector_returns = grid.merge(sector_returns, how="left", on=["date", "sector"]).sort_values(["sector", "date"])
print(f"Sector return rows: {len(sector_returns):,}")

# -----------------------------
# 10) Pull FRED macro data
# -----------------------------
print("Pulling FRED macro data...")
fred_api_key = getpass("Enter FRED API key: ").strip()
fred_client = Fred(api_key=fred_api_key)

fred_start = pd.to_datetime(START_DATE) - pd.DateOffset(months=24)
fred_end = pd.to_datetime(END_DATE)

wanted_series = ["DGS10", "DGS2", "UNRATE", "INDPRO"]
fred_dict = {}

for series_code in wanted_series:
    try:
        s = fred_client.get_series(series_code, observation_start=fred_start, observation_end=fred_end)
        s = pd.Series(s, name=series_code)
        fred_dict[series_code] = s
        print(f"Loaded FRED series: {series_code}")
    except Exception as e:
        print(f"Could not pull FRED series {series_code}: {e}")

if len(fred_dict) > 0:
    fred = pd.concat(fred_dict.values(), axis=1).reset_index()
    fred = fred.rename(columns={"index": "date"})
    fred["date"] = pd.to_datetime(fred["date"]) + pd.offsets.MonthEnd(0)
    fred = fred.sort_values("date").groupby("date", as_index=False).last()

    if {"DGS10", "DGS2"}.issubset(fred.columns):
        fred["yield_spread_10y_2y"] = fred["DGS10"] - fred["DGS2"]
    else:
        fred["yield_spread_10y_2y"] = np.nan
else:
    fred = pd.DataFrame(columns=["date", "DGS10", "DGS2", "UNRATE", "INDPRO", "yield_spread_10y_2y"])

def label_macro_regime(spread):
    if pd.isna(spread):
        return np.nan
    if spread < 0:
        return "Contraction"
    elif spread < 1:
        return "Neutral"
    else:
        return "Expansion"

if len(fred) > 0:
    fred["macro_regime"] = fred["yield_spread_10y_2y"].apply(label_macro_regime)
else:
    fred["macro_regime"] = pd.Series(dtype="object")

# -----------------------------
# 11) Signal engine
# -----------------------------
def build_signals(sector_df, lookback_months=6, skip_months=1, top_n=3, bottom_n=3, use_macro_overlay=False):
    df = sector_df.copy().sort_values(["sector", "date"])

    df["momentum_raw"] = (
        df.groupby("sector")["sector_ret"]
        .transform(lambda x: x.shift(skip_months).rolling(lookback_months).apply(compute_compound_return, raw=False))
    )

    df["rank"] = df.groupby("date")["momentum_raw"].rank(ascending=False, method="first")
    df["n_sectors"] = df.groupby("date")["sector"].transform("count")

    df = zscore_by_date(df, "momentum_raw")
    df = df.rename(columns={"zscore": "momentum_z"})

    df["final_score"] = df["momentum_z"]
    df["macro_adjustment"] = 0.0

    if use_macro_overlay and (not fred.empty) and ("yield_spread_10y_2y" in fred.columns):
        df = df.merge(fred[["date", "yield_spread_10y_2y", "macro_regime"]], on="date", how="left")

        def macro_boost(row):
            reg = row["macro_regime"]
            sec = row["sector"]
            if pd.isna(reg):
                return 0.0
            if reg == "Expansion" and sec in CYCLICALS:
                return 0.25
            if reg == "Contraction" and sec in DEFENSIVES:
                return 0.25
            return 0.0

        df["macro_adjustment"] = df.apply(macro_boost, axis=1)
        df["final_score"] = df["momentum_z"] + df["macro_adjustment"]
        df["rank"] = df.groupby("date")["final_score"].rank(ascending=False, method="first")
    else:
        df["macro_regime"] = np.nan
        df["yield_spread_10y_2y"] = np.nan

    df["signal"] = df.apply(
        lambda r: traffic_light_from_rank(
            r["rank"],
            int(r["n_sectors"]) if not pd.isna(r["n_sectors"]) else 11,
            top_n,
            bottom_n
        ),
        axis=1
    )

    df["fwd_1m_ret"] = df.groupby("sector")["sector_ret"].shift(-1)
    return df

# -----------------------------
# 12) Backtest engine
# -----------------------------
def run_backtest(signal_df, top_n=3):
    df = signal_df.copy()
    valid = df.dropna(subset=["rank", "fwd_1m_ret"]).copy()

    valid["selected"] = valid["rank"] <= top_n

    port = valid[valid["selected"]].groupby("date", as_index=False).agg(
        strategy_ret=("fwd_1m_ret", "mean")
    )

    bench = valid.groupby("date", as_index=False).agg(
        benchmark_ret=("fwd_1m_ret", "mean")
    )

    green = valid[valid["signal"] == "Green"].groupby("date", as_index=False).agg(
        green_ret=("fwd_1m_ret", "mean")
    )

    red = valid[valid["signal"] == "Red"].groupby("date", as_index=False).agg(
        red_ret=("fwd_1m_ret", "mean")
    )

    out = (
        port.merge(bench, on="date", how="outer")
            .merge(green, on="date", how="left")
            .merge(red, on="date", how="left")
            .sort_values("date")
            .reset_index(drop=True)
    )

    out["strategy_cum"] = (1 + out["strategy_ret"].fillna(0)).cumprod()
    out["benchmark_cum"] = (1 + out["benchmark_ret"].fillna(0)).cumprod()
    out["spread_ret"] = out["green_ret"] - out["red_ret"]
    out["spread_cum"] = (1 + out["spread_ret"].fillna(0)).cumprod()

    return out

def summarize_performance(bt):
    return pd.DataFrame({
        "Metric": [
            "Annualized Return",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Max Drawdown",
            "Hit Rate vs Benchmark"
        ],
        "Strategy": [
            annualized_return(bt["strategy_ret"]),
            annualized_vol(bt["strategy_ret"]),
            sharpe_ratio(bt["strategy_ret"]),
            max_drawdown(bt["strategy_ret"]),
            safe_hit_rate(bt["strategy_ret"], bt["benchmark_ret"])
        ],
        "Benchmark": [
            annualized_return(bt["benchmark_ret"]),
            annualized_vol(bt["benchmark_ret"]),
            sharpe_ratio(bt["benchmark_ret"]),
            max_drawdown(bt["benchmark_ret"]),
            np.nan
        ],
        "Green-Red Spread": [
            annualized_return(bt["spread_ret"]),
            annualized_vol(bt["spread_ret"]),
            sharpe_ratio(bt["spread_ret"]),
            max_drawdown(bt["spread_ret"]),
            np.nan
        ]
    })

def display_perf_table(perf):
    perf_fmt = perf.copy()
    pct_metrics = {"Annualized Return", "Annualized Volatility", "Max Drawdown", "Hit Rate vs Benchmark"}
    for col in ["Strategy", "Benchmark", "Green-Red Spread"]:
        perf_fmt[col] = perf_fmt.apply(
            lambda row: f"{row[col]:.2%}" if (row["Metric"] in pct_metrics and pd.notna(row[col]))
            else (f"{row[col]:.2f}" if pd.notna(row[col]) else ""),
            axis=1
        )
    display(perf_fmt)

# -----------------------------
# 13) Initial build
# -----------------------------
signals = build_signals(
    sector_returns,
    lookback_months=DEFAULT_LOOKBACK_MONTHS,
    skip_months=DEFAULT_SKIP_MONTHS,
    top_n=DEFAULT_TOP_N,
    bottom_n=DEFAULT_BOTTOM_N,
    use_macro_overlay=USE_MACRO_OVERLAY
)
backtest = run_backtest(signals, top_n=DEFAULT_TOP_N)

# -----------------------------
# 14) Dashboard renderer
# -----------------------------
def render_dashboard(start_date, end_date, lookback_months, skip_months, top_n, bottom_n, use_macro_overlay):
    clear_output(wait=True)

    sig = build_signals(
        sector_returns,
        lookback_months=int(lookback_months),
        skip_months=int(skip_months),
        top_n=int(top_n),
        bottom_n=int(bottom_n),
        use_macro_overlay=bool(use_macro_overlay)
    )

    bt = run_backtest(sig, top_n=int(top_n))

    start_date = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end_date = pd.to_datetime(end_date) + pd.offsets.MonthEnd(0)

    sig_f = sig[(sig["date"] >= start_date) & (sig["date"] <= end_date)].copy()
    bt_f = bt[(bt["date"] >= start_date) & (bt["date"] <= end_date)].copy()

    if len(sig_f) == 0 or len(bt_f) == 0:
        print("No data in selected range.")
        return

    current_date = sig_f["date"].max()
    current = sig_f[sig_f["date"] == current_date].sort_values("rank").copy()
    perf = summarize_performance(bt_f)

    bt_f["strategy_cum"] = (1 + bt_f["strategy_ret"].fillna(0)).cumprod()
    bt_f["benchmark_cum"] = (1 + bt_f["benchmark_ret"].fillna(0)).cumprod()
    bt_f["spread_cum"] = (1 + bt_f["spread_ret"].fillna(0)).cumprod()

    print("=" * 78)
    print("SECTOR ROTATION DASHBOARD")
    print("=" * 78)
    print(f"Date range: {start_date.date()} to {end_date.date()}")
    print(f"Lookback months: {lookback_months} | Skip months: {skip_months} | Top N: {top_n} | Bottom N: {bottom_n} | Macro overlay: {use_macro_overlay}")
    print(f"Latest signal date: {current_date.date()}")
    print("=" * 78)

    print("\nCURRENT MONTH SIGNAL TABLE")
    show_cols = ["date", "sector", "momentum_raw", "momentum_z", "macro_adjustment", "final_score", "rank", "signal", "n_stocks"]
    current_show = current[show_cols].copy()
    current_show["momentum_raw"] = current_show["momentum_raw"].round(4)
    current_show["momentum_z"] = current_show["momentum_z"].round(3)
    current_show["macro_adjustment"] = current_show["macro_adjustment"].round(3)
    current_show["final_score"] = current_show["final_score"].round(3)
    display(current_show.reset_index(drop=True))

    print("\nBACKTEST SUMMARY")
    display_perf_table(perf)

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["strategy_cum"], mode="lines", name="Strategy"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["benchmark_cum"], mode="lines", name="Benchmark"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["spread_cum"], mode="lines", name="Green-Red Spread"))
    fig1.update_layout(title="Cumulative Performance", xaxis_title="Date", yaxis_title="Growth of $1", template="plotly_white", width=1000, height=500)
    fig1.show()

    signal_to_num = {"Red": -1, "Yellow": 0, "Green": 1}
    hm = sig_f[["date", "sector", "signal"]].copy()
    hm["signal_num"] = hm["signal"].map(signal_to_num)
    heat = hm.pivot(index="sector", columns="date", values="signal_num").reindex(all_sectors)

    fig2 = px.imshow(heat, aspect="auto", title="Traffic Light Heatmap", color_continuous_scale=["red", "yellow", "green"], zmin=-1, zmax=1)
    fig2.update_layout(width=1200, height=500)
    fig2.show()

    fig3 = px.bar(current.sort_values("rank"), x="sector", y="momentum_raw", color="signal",
                  title=f"Current Sector Momentum ({current_date.date()})",
                  hover_data=["rank", "final_score", "n_stocks"])
    fig3.update_layout(template="plotly_white", width=1000, height=500)
    fig3.show()

# -----------------------------
# 15) Widgets
# -----------------------------
available_dates = pd.Series(sorted(sector_returns["date"].dropna().unique()))
min_date = available_dates.min().date()
max_date = available_dates.max().date()

display(HTML("<h2>Sector Rotation Model Dashboard</h2>"))

start_picker = widgets.DatePicker(description="Start", value=min_date)
end_picker = widgets.DatePicker(description="End", value=max_date)
lookback_slider = widgets.IntSlider(value=DEFAULT_LOOKBACK_MONTHS, min=3, max=12, step=1, description="Lookback")
skip_slider = widgets.IntSlider(value=DEFAULT_SKIP_MONTHS, min=0, max=2, step=1, description="Skip")
topn_slider = widgets.IntSlider(value=DEFAULT_TOP_N, min=1, max=5, step=1, description="Top N")
bottomn_slider = widgets.IntSlider(value=DEFAULT_BOTTOM_N, min=1, max=5, step=1, description="Bottom N")
macro_toggle = widgets.Checkbox(value=USE_MACRO_OVERLAY, description="Macro overlay")
run_button = widgets.Button(description="Run Dashboard", button_style="success")
output = widgets.Output()

def on_run_clicked(_):
    with output:
        render_dashboard(
            start_date=start_picker.value,
            end_date=end_picker.value,
            lookback_months=lookback_slider.value,
            skip_months=skip_slider.value,
            top_n=topn_slider.value,
            bottom_n=bottomn_slider.value,
            use_macro_overlay=macro_toggle.value
        )

run_button.on_click(on_run_clicked)

controls = widgets.VBox([
    widgets.HBox([start_picker, end_picker]),
    widgets.HBox([lookback_slider, skip_slider]),
    widgets.HBox([topn_slider, bottomn_slider]),
    widgets.HBox([macro_toggle, run_button]),
])

display(controls, output)

with output:
    render_dashboard(
        start_date=start_picker.value,
        end_date=end_picker.value,
        lookback_months=lookback_slider.value,
        skip_months=skip_slider.value,
        top_n=topn_slider.value,
        bottom_n=bottomn_slider.value,
        use_macro_overlay=macro_toggle.value
    )

In [ ]:
# ============================================================
# BLUE EAGLE SECTOR ROTATION DASHBOARD
# Google Colab | One-cell version
# Monthly rebalance | Momentum MVP | WRDS + Compustat + FRED
# ============================================================

# -----------------------------
# 0) Install packages
# -----------------------------
import sys, subprocess, pkgutil

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["wrds", "fredapi", "plotly", "ipywidgets"]
for pkg in required:
    if pkgutil.find_loader(pkg) is None:
        pip_install(pkg)

# -----------------------------
# 1) Imports
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import wrds
from fredapi import Fred
from getpass import getpass
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# -----------------------------
# 2) User parameters
# -----------------------------
START_DATE = "2010-01-01"
END_DATE   = "2025-12-31"

DEFAULT_LOOKBACK_MONTHS = 6
DEFAULT_SKIP_MONTHS     = 1
DEFAULT_TOP_N           = 3
DEFAULT_BOTTOM_N        = 3

USE_MACRO_OVERLAY = False
SAVE_CSV_OUTPUTS  = False

# -----------------------------
# 3) Helper functions
# -----------------------------
GICS_MAP = {
    "10": "Energy",
    "15": "Materials",
    "20": "Industrials",
    "25": "Consumer Discretionary",
    "30": "Consumer Staples",
    "35": "Health Care",
    "40": "Financials",
    "45": "Information Technology",
    "50": "Communication Services",
    "55": "Utilities",
    "60": "Real Estate"
}

CYCLICALS = {
    "Energy", "Materials", "Industrials", "Consumer Discretionary",
    "Financials", "Information Technology", "Communication Services", "Real Estate"
}

DEFENSIVES = {"Consumer Staples", "Health Care", "Utilities"}

def coerce_numeric(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def annualized_return(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    cum = (1 + r).prod()
    years = len(r) / 12.0
    if years <= 0 or cum <= 0:
        return np.nan
    return cum ** (1 / years) - 1

def annualized_vol(r):
    r = pd.Series(r).dropna()
    if len(r) < 2:
        return np.nan
    return r.std() * np.sqrt(12)

def sharpe_ratio(r, rf=0.0):
    ar = annualized_return(r)
    av = annualized_vol(r)
    if pd.isna(ar) or pd.isna(av) or av == 0:
        return np.nan
    return (ar - rf) / av

def max_drawdown(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    wealth = (1 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1
    return dd.min()

def compute_compound_return(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return np.prod(1 + x) - 1

def traffic_light_from_rank(rank, n_sectors, top_n=3, bottom_n=3):
    if pd.isna(rank):
        return np.nan
    if rank <= top_n:
        return "Green"
    elif rank > n_sectors - bottom_n:
        return "Red"
    else:
        return "Yellow"

def zscore_by_date(df, value_col):
    out = df.copy()
    def _z(x):
        sd = x.std(ddof=0)
        if pd.isna(sd) or sd == 0:
            return pd.Series(np.nan, index=x.index)
        return (x - x.mean()) / sd
    out["zscore"] = out.groupby("date")[value_col].transform(_z)
    return out

def safe_hit_rate(strategy, benchmark):
    s = pd.Series(strategy)
    b = pd.Series(benchmark)
    mask = s.notna() & b.notna()
    if mask.sum() == 0:
        return np.nan
    return (s[mask] > b[mask]).mean()

# -----------------------------
# 4) Connect to WRDS
# -----------------------------
print("Enter your WRDS credentials.")
wrds_username = input("WRDS username: ").strip()
wrds_password = getpass("WRDS password: ")

db = wrds.Connection(wrds_username=wrds_username, password=wrds_password)

# -----------------------------
# 5) Pull CRSP monthly stock data
# -----------------------------
print("Pulling CRSP monthly data...")

crsp_sql = f"""
select
    a.permno,
    a.permco,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    b.shrcd,
    b.exchcd,
    b.ticker,
    b.comnam

from crsp.msf as a
join crsp.msenames as b
    on a.permno = b.permno
   and b.namedt <= a.date
   and a.date <= b.nameendt
where a.date between '{START_DATE}' and '{END_DATE}'
  and b.shrcd in (10, 11)
  and b.exchcd in (1, 2, 3)
"""

crsp = db.raw_sql(crsp_sql, date_cols=["date"])
crsp = coerce_numeric(crsp, ["ret", "prc", "shrout", "shrcd", "exchcd"])
crsp = crsp.dropna(subset=["date", "permno", "ret", "prc", "shrout"]).copy()
crsp["mkt_cap"] = crsp["prc"].abs() * crsp["shrout"]
crsp["date"] = pd.to_datetime(crsp["date"]) + pd.offsets.MonthEnd(0)

print(f"CRSP rows: {len(crsp):,}")

# -----------------------------
# 6) Pull CCM link table
# -----------------------------
print("Pulling CRSP-Compustat link table...")

ccm_sql = """
select
    gvkey,
    lpermno as permno,
    linktype,
    linkprim,
    linkdt,
    linkenddt
from crsp.ccmxpf_linktable
where lpermno is not null
  and linktype in ('LU','LC','LS')
  and linkprim in ('P','C')
"""
ccm = db.raw_sql(ccm_sql, date_cols=["linkdt", "linkenddt"])
ccm["linkenddt"] = ccm["linkenddt"].fillna(pd.Timestamp("2100-12-31"))

# -----------------------------
# 7) Pull Compustat GICS sector
# -----------------------------
print("Pulling Compustat GICS sectors...")

comp_sql = """
select
    gvkey,
    conm,
    gsector
from comp.company
where gsector is not null
"""
comp = db.raw_sql(comp_sql)
comp["gsector"] = comp["gsector"].astype(str).str.strip()

# -----------------------------
# 8) Merge CRSP -> CCM -> Compustat
# -----------------------------
print("Merging CRSP, CCM, and Compustat...")

merged = crsp.merge(ccm, how="left", on="permno")
merged = merged[
    (merged["date"] >= merged["linkdt"]) &
    (merged["date"] <= merged["linkenddt"])
].copy()

merged = merged.sort_values(["permno", "date", "gvkey"]).drop_duplicates(["permno", "date"], keep="first")
merged = merged.merge(comp[["gvkey", "gsector"]], how="left", on="gvkey")

merged["gsector"] = merged["gsector"].astype(str).str.strip()
merged["sector"] = merged["gsector"].map(GICS_MAP)
merged = merged[merged["sector"].notna()].copy()

print(f"Merged mapped rows: {len(merged):,}")

# -----------------------------
# 9) Build value-weighted sector returns
# -----------------------------
print("Building value-weighted sector returns...")

merged = merged.sort_values(["permno", "date"]).copy()
merged["lag_mkt_cap"] = merged.groupby("permno")["mkt_cap"].shift(1)
merged = merged.dropna(subset=["lag_mkt_cap", "ret"]).copy()
merged = merged[merged["lag_mkt_cap"] > 0].copy()

sector_totals = merged.groupby(["date", "sector"], as_index=False)["lag_mkt_cap"].sum()
sector_totals = sector_totals.rename(columns={"lag_mkt_cap": "sector_lag_cap"})

merged = merged.merge(sector_totals, on=["date", "sector"], how="left")
merged["weight"] = merged["lag_mkt_cap"] / merged["sector_lag_cap"]

sector_returns = (
    merged.groupby(["date", "sector"], as_index=False)
    .apply(lambda x: pd.Series({
        "sector_ret": np.sum(x["weight"] * x["ret"]),
        "n_stocks": x["permno"].nunique(),
        "sector_lag_cap": x["sector_lag_cap"].iloc[0]
    }))
    .reset_index(drop=True)
)

all_dates = pd.date_range(sector_returns["date"].min(), sector_returns["date"].max(), freq="M")
all_sectors = list(GICS_MAP.values())
grid = pd.MultiIndex.from_product([all_dates, all_sectors], names=["date", "sector"]).to_frame(index=False)

sector_returns = grid.merge(sector_returns, how="left", on=["date", "sector"]).sort_values(["sector", "date"])

print(f"Sector return rows: {len(sector_returns):,}")

# -----------------------------
# 10) Pull FRED macro data
# -----------------------------
print("Pulling FRED macro data...")

FRED_API_KEY = getpass("Enter FRED API key: ").strip()
fred_client = Fred(api_key=FRED_API_KEY)

fred_start = pd.to_datetime(START_DATE) - pd.DateOffset(months=24)
fred_end = pd.to_datetime(END_DATE)

wanted_series = ["DGS10", "DGS2", "UNRATE", "INDPRO"]
fred_dict = {}

for series_code in wanted_series:
    try:
        s = fred_client.get_series(series_code, observation_start=fred_start, observation_end=fred_end)
        s = pd.Series(s, name=series_code)
        fred_dict[series_code] = s
        print(f"Loaded FRED series: {series_code}")
    except Exception as e:
        print(f"Could not pull FRED series {series_code}: {e}")

if len(fred_dict) > 0:
    fred = pd.concat(fred_dict.values(), axis=1).reset_index()
    fred = fred.rename(columns={"index": "date"})
    fred["date"] = pd.to_datetime(fred["date"]) + pd.offsets.MonthEnd(0)
    fred = fred.sort_values("date").groupby("date", as_index=False).last()

    if {"DGS10", "DGS2"}.issubset(fred.columns):
        fred["yield_spread_10y_2y"] = fred["DGS10"] - fred["DGS2"]
    else:
        fred["yield_spread_10y_2y"] = np.nan
else:
    fred = pd.DataFrame(columns=["date", "DGS10", "DGS2", "UNRATE", "INDPRO", "yield_spread_10y_2y"])

def label_macro_regime(spread):
    if pd.isna(spread):
        return np.nan
    if spread < 0:
        return "Contraction"
    elif spread < 1:
        return "Neutral"
    else:
        return "Expansion"

if len(fred) > 0:
    fred["macro_regime"] = fred["yield_spread_10y_2y"].apply(label_macro_regime)
else:
    fred["macro_regime"] = pd.Series(dtype="object")

# -----------------------------
# 11) Signal engine
# -----------------------------
def build_signals(sector_df, lookback_months=6, skip_months=1, top_n=3, bottom_n=3, use_macro_overlay=False):
    df = sector_df.copy().sort_values(["sector", "date"])

    df["momentum_raw"] = (
        df.groupby("sector")["sector_ret"]
        .transform(lambda x: x.shift(skip_months).rolling(lookback_months).apply(compute_compound_return, raw=False))
    )

    df["rank"] = df.groupby("date")["momentum_raw"].rank(ascending=False, method="first")
    df["n_sectors"] = df.groupby("date")["sector"].transform("count")

    df = zscore_by_date(df, "momentum_raw")
    df = df.rename(columns={"zscore": "momentum_z"})

    df["final_score"] = df["momentum_z"]
    df["macro_adjustment"] = 0.0
    df["macro_regime"] = np.nan
    df["yield_spread_10y_2y"] = np.nan

    if use_macro_overlay and (not fred.empty) and ("yield_spread_10y_2y" in fred.columns):
        df = df.merge(fred[["date", "yield_spread_10y_2y", "macro_regime"]], on="date", how="left")

        def macro_boost(row):
            reg = row["macro_regime"]
            sec = row["sector"]
            if pd.isna(reg):
                return 0.0
            if reg == "Expansion" and sec in CYCLICALS:
                return 0.25
            if reg == "Contraction" and sec in DEFENSIVES:
                return 0.25
            return 0.0

        df["macro_adjustment"] = df.apply(macro_boost, axis=1)
        df["final_score"] = df["momentum_z"] + df["macro_adjustment"]
        df["rank"] = df.groupby("date")["final_score"].rank(ascending=False, method="first")

    df["signal"] = df.apply(
        lambda r: traffic_light_from_rank(
            r["rank"],
            int(r["n_sectors"]) if not pd.isna(r["n_sectors"]) else 11,
            top_n,
            bottom_n
        ),
        axis=1
    )

    df["fwd_1m_ret"] = df.groupby("sector")["sector_ret"].shift(-1)
    return df

# -----------------------------
# 12) Backtest engine
# -----------------------------
def run_backtest(signal_df, top_n=3):
    df = signal_df.copy()
    valid = df.dropna(subset=["rank", "fwd_1m_ret"]).copy()

    valid["selected"] = valid["rank"] <= top_n

    port = valid[valid["selected"]].groupby("date", as_index=False).agg(
        strategy_ret=("fwd_1m_ret", "mean")
    )

    bench = valid.groupby("date", as_index=False).agg(
        benchmark_ret=("fwd_1m_ret", "mean")
    )

    green = valid[valid["signal"] == "Green"].groupby("date", as_index=False).agg(
        green_ret=("fwd_1m_ret", "mean")
    )

    red = valid[valid["signal"] == "Red"].groupby("date", as_index=False).agg(
        red_ret=("fwd_1m_ret", "mean")
    )

    out = (
        port.merge(bench, on="date", how="outer")
            .merge(green, on="date", how="left")
            .merge(red, on="date", how="left")
            .sort_values("date")
            .reset_index(drop=True)
    )

    out["strategy_cum"] = (1 + out["strategy_ret"].fillna(0)).cumprod()
    out["benchmark_cum"] = (1 + out["benchmark_ret"].fillna(0)).cumprod()
    out["green_cum"] = (1 + out["green_ret"].fillna(0)).cumprod()
    out["red_cum"] = (1 + out["red_ret"].fillna(0)).cumprod()
    out["spread_ret"] = out["green_ret"] - out["red_ret"]
    out["spread_cum"] = (1 + out["spread_ret"].fillna(0)).cumprod()

    return out

def summarize_performance(bt):
    return pd.DataFrame({
        "Metric": [
            "Annualized Return",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Max Drawdown",
            "Hit Rate vs Benchmark"
        ],
        "Strategy": [
            annualized_return(bt["strategy_ret"]),
            annualized_vol(bt["strategy_ret"]),
            sharpe_ratio(bt["strategy_ret"]),
            max_drawdown(bt["strategy_ret"]),
            safe_hit_rate(bt["strategy_ret"], bt["benchmark_ret"])
        ],
        "Benchmark": [
            annualized_return(bt["benchmark_ret"]),
            annualized_vol(bt["benchmark_ret"]),
            sharpe_ratio(bt["benchmark_ret"]),
            max_drawdown(bt["benchmark_ret"]),
            np.nan
        ],
        "Green-Red Spread": [
            annualized_return(bt["spread_ret"]),
            annualized_vol(bt["spread_ret"]),
            sharpe_ratio(bt["spread_ret"]),
            max_drawdown(bt["spread_ret"]),
            np.nan
        ]
    })

def display_perf_table(perf):
    perf_fmt = perf.copy()
    pct_metrics = {"Annualized Return", "Annualized Volatility", "Max Drawdown", "Hit Rate vs Benchmark"}

    for col in ["Strategy", "Benchmark", "Green-Red Spread"]:
        perf_fmt[col] = perf_fmt.apply(
            lambda row: f"{row[col]:.2%}" if (row["Metric"] in pct_metrics and pd.notna(row[col]))
            else (f"{row[col]:.2f}" if pd.notna(row[col]) else ""),
            axis=1
        )
    display(perf_fmt)

# -----------------------------
# 13) Initial build
# -----------------------------
signals = build_signals(
    sector_returns,
    lookback_months=DEFAULT_LOOKBACK_MONTHS,
    skip_months=DEFAULT_SKIP_MONTHS,
    top_n=DEFAULT_TOP_N,
    bottom_n=DEFAULT_BOTTOM_N,
    use_macro_overlay=USE_MACRO_OVERLAY
)

backtest = run_backtest(signals, top_n=DEFAULT_TOP_N)
summary = summarize_performance(backtest)

# -----------------------------
# 14) Dashboard renderer
# -----------------------------
def render_dashboard(start_date, end_date, lookback_months, skip_months, top_n, bottom_n, use_macro_overlay):
    clear_output(wait=True)

    sig = build_signals(
        sector_returns,
        lookback_months=int(lookback_months),
        skip_months=int(skip_months),
        top_n=int(top_n),
        bottom_n=int(bottom_n),
        use_macro_overlay=bool(use_macro_overlay)
    )

    bt = run_backtest(sig, top_n=int(top_n))

    start_date = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end_date = pd.to_datetime(end_date) + pd.offsets.MonthEnd(0)

    sig_f = sig[(sig["date"] >= start_date) & (sig["date"] <= end_date)].copy()
    bt_f = bt[(bt["date"] >= start_date) & (bt["date"] <= end_date)].copy()

    if len(sig_f) == 0 or len(bt_f) == 0:
        print("No data in selected range.")
        return

    current_date = sig_f["date"].max()
    current = sig_f[sig_f["date"] == current_date].sort_values("rank").copy()
    perf = summarize_performance(bt_f)

    bt_f["strategy_cum"] = (1 + bt_f["strategy_ret"].fillna(0)).cumprod()
    bt_f["benchmark_cum"] = (1 + bt_f["benchmark_ret"].fillna(0)).cumprod()
    bt_f["spread_cum"] = (1 + bt_f["spread_ret"].fillna(0)).cumprod()

    print("=" * 78)
    print("SECTOR ROTATION DASHBOARD")
    print("=" * 78)
    print(f"Date range: {start_date.date()} to {end_date.date()}")
    print(f"Lookback months: {lookback_months} | Skip months: {skip_months} | Top N: {top_n} | Bottom N: {bottom_n} | Macro overlay: {use_macro_overlay}")
    print(f"Latest signal date: {current_date.date()}")
    print("=" * 78)

    print("\nCURRENT MONTH SIGNAL TABLE")
    show_cols = [
        "date", "sector", "momentum_raw", "momentum_z",
        "macro_adjustment", "final_score", "rank", "signal", "n_stocks"
    ]
    current_show = current[show_cols].copy()
    current_show["momentum_raw"] = current_show["momentum_raw"].round(4)
    current_show["momentum_z"] = current_show["momentum_z"].round(3)
    current_show["macro_adjustment"] = current_show["macro_adjustment"].round(3)
    current_show["final_score"] = current_show["final_score"].round(3)
    display(current_show.reset_index(drop=True))

    print("\nBACKTEST SUMMARY")
    display_perf_table(perf)

    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["strategy_cum"], mode="lines", name="Strategy"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["benchmark_cum"], mode="lines", name="Benchmark"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["spread_cum"], mode="lines", name="Green-Red Spread"))
    fig1.update_layout(
        title="Cumulative Performance",
        xaxis_title="Date",
        yaxis_title="Growth of $1",
        template="plotly_white",
        width=1000,
        height=500
    )
    fig1.show()

    signal_to_num = {"Red": -1, "Yellow": 0, "Green": 1}
    hm = sig_f[["date", "sector", "signal"]].copy()
    hm["signal_num"] = hm["signal"].map(signal_to_num)
    heat = hm.pivot(index="sector", columns="date", values="signal_num").reindex(all_sectors)

    fig2 = px.imshow(
        heat,
        aspect="auto",
        title="Traffic Light Heatmap",
        color_continuous_scale=["red", "yellow", "green"],
        zmin=-1,
        zmax=1
    )
    fig2.update_layout(width=1200, height=500)
    fig2.show()

    fig3 = px.bar(
        current.sort_values("rank"),
        x="sector",
        y="momentum_raw",
        color="signal",
        title=f"Current Sector Momentum ({current_date.date()})",
        hover_data=["rank", "final_score", "n_stocks"]
    )
    fig3.update_layout(template="plotly_white", width=1000, height=500)
    fig3.show()

    if (not fred.empty) and ("yield_spread_10y_2y" in fred.columns):
        fred_f = fred[(fred["date"] >= start_date) & (fred["date"] <= end_date)].copy()
        if len(fred_f) > 0:
            fig4 = go.Figure()
            fig4.add_trace(go.Scatter(
                x=fred_f["date"],
                y=fred_f["yield_spread_10y_2y"],
                mode="lines",
                name="10Y-2Y Spread"
            ))
            fig4.update_layout(
                title="Macro Context: 10Y - 2Y Treasury Spread",
                xaxis_title="Date",
                yaxis_title="Spread",
                template="plotly_white",
                width=1000,
                height=400
            )
            fig4.show()

    if SAVE_CSV_OUTPUTS:
        sig_f.to_csv("sector_signals_filtered.csv", index=False)
        bt_f.to_csv("sector_backtest_filtered.csv", index=False)
        current_show.to_csv("current_signal_table.csv", index=False)
        print("\nCSV files saved to current Colab working directory.")

# -----------------------------
# 15) Widgets
# -----------------------------
available_dates = pd.Series(sorted(sector_returns["date"].dropna().unique()))
min_date = available_dates.min().date()
max_date = available_dates.max().date()

display(HTML("<h2>Sector Rotation Model Dashboard</h2>"))

start_picker = widgets.DatePicker(description="Start", value=min_date)
end_picker = widgets.DatePicker(description="End", value=max_date)

lookback_slider = widgets.IntSlider(value=DEFAULT_LOOKBACK_MONTHS, min=3, max=12, step=1, description="Lookback")
skip_slider = widgets.IntSlider(value=DEFAULT_SKIP_MONTHS, min=0, max=2, step=1, description="Skip")
topn_slider = widgets.IntSlider(value=DEFAULT_TOP_N, min=1, max=5, step=1, description="Top N")
bottomn_slider = widgets.IntSlider(value=DEFAULT_BOTTOM_N, min=1, max=5, step=1, description="Bottom N")
macro_toggle = widgets.Checkbox(value=USE_MACRO_OVERLAY, description="Macro overlay")
run_button = widgets.Button(description="Run Dashboard", button_style="success")
output = widgets.Output()

def on_run_clicked(_):
    with output:
        render_dashboard(
            start_date=start_picker.value,
            end_date=end_picker.value,
            lookback_months=lookback_slider.value,
            skip_months=skip_slider.value,
            top_n=topn_slider.value,
            bottom_n=bottomn_slider.value,
            use_macro_overlay=macro_toggle.value
        )

run_button.on_click(on_run_clicked)

controls = widgets.VBox([
    widgets.HBox([start_picker, end_picker]),
    widgets.HBox([lookback_slider, skip_slider]),
    widgets.HBox([topn_slider, bottomn_slider]),
    widgets.HBox([macro_toggle, run_button]),
])

display(controls, output)

with output:
    render_dashboard(
        start_date=start_picker.value,
        end_date=end_picker.value,
        lookback_months=lookback_slider.value,
        skip_months=skip_slider.value,
        top_n=topn_slider.value,
        bottom_n=bottomn_slider.value,
        use_macro_overlay=macro_toggle.value
    )

In [ ]:
# ============================================================
# SECTOR ROTATION DASHBOARD (Google Colab, one-cell version)
# Monthly rebalance | Momentum MVP | WRDS + Compustat + FRED
# ============================================================

# -----------------------------
# 0) Install packages
# -----------------------------
import sys, subprocess, pkgutil

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["wrds", "pandas_datareader", "plotly", "ipywidgets"]
for pkg in required:
    if pkgutil.find_loader(pkg) is None:
        pip_install(pkg)

# -----------------------------
# 1) Imports
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import wrds
from getpass import getpass
from pandas_datareader import data as web
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# 2) User parameters
# -----------------------------
# You can change these defaults later in the dashboard controls too
START_DATE = "2010-01-01"
END_DATE   = "2025-12-31"

DEFAULT_LOOKBACK_MONTHS = 6
DEFAULT_SKIP_MONTHS     = 1
DEFAULT_TOP_N           = 3
DEFAULT_BOTTOM_N        = 3

USE_MACRO_OVERLAY = False   # phase 1 = False
SAVE_CSV_OUTPUTS  = False   # set True if you want CSVs written in Colab

# -----------------------------
# 3) Helper functions
# -----------------------------
GICS_MAP = {
    "10": "Energy",
    "15": "Materials",
    "20": "Industrials",
    "25": "Consumer Discretionary",
    "30": "Consumer Staples",
    "35": "Health Care",
    "40": "Financials",
    "45": "Information Technology",
    "50": "Communication Services",
    "55": "Utilities",
    "60": "Real Estate"
}

CYCLICALS  = {"Energy", "Materials", "Industrials", "Consumer Discretionary", "Financials", "Information Technology", "Communication Services", "Real Estate"}
DEFENSIVES = {"Consumer Staples", "Health Care", "Utilities"}

def coerce_numeric(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def annualized_return(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    cum = (1 + r).prod()
    years = len(r) / 12.0
    if years <= 0 or cum <= 0:
        return np.nan
    return cum ** (1 / years) - 1

def annualized_vol(r):
    r = pd.Series(r).dropna()
    if len(r) < 2:
        return np.nan
    return r.std() * np.sqrt(12)

def sharpe_ratio(r, rf=0.0):
    ar = annualized_return(r)
    av = annualized_vol(r)
    if pd.isna(ar) or pd.isna(av) or av == 0:
        return np.nan
    return (ar - rf) / av

def max_drawdown(r):
    r = pd.Series(r).dropna()
    if len(r) == 0:
        return np.nan
    wealth = (1 + r).cumprod()
    peak = wealth.cummax()
    dd = wealth / peak - 1
    return dd.min()

def compute_compound_return(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return np.prod(1 + x) - 1

def traffic_light_from_rank(rank, n_sectors, top_n=3, bottom_n=3):
    if pd.isna(rank):
        return np.nan
    if rank <= top_n:
        return "Green"
    elif rank > n_sectors - bottom_n:
        return "Red"
    else:
        return "Yellow"

def zscore_by_date(df, value_col):
    out = df.copy()
    out["zscore"] = out.groupby("date")[value_col].transform(
        lambda x: (x - x.mean()) / x.std(ddof=0) if x.std(ddof=0) not in [0, np.nan] else np.nan
    )
    return out

# -----------------------------
# 4) Connect to WRDS
# -----------------------------
print("Enter your WRDS credentials.")
wrds_username = input("WRDS username: ").strip()
wrds_password = getpass("WRDS password: ")

db = wrds.Connection(wrds_username=wrds_username, password=wrds_password)

# -----------------------------
# 5) Pull CRSP monthly stock data
# -----------------------------
# Monthly stock file + name history filters:
# common stocks, NYSE/AMEX/Nasdaq
# Note: ret can contain non-numeric codes, so we coerce later.

crsp_sql = f"""
select
    a.permno,
    a.permco,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    b.shrcd,
    b.exchcd,
    b.ticker,
    b.comnam
from crsp.msf as a
join crsp.msenames as b
    on a.permno = b.permno
   and b.namedt <= a.date
   and a.date <= b.nameendt
where a.date between '{START_DATE}' and '{END_DATE}'
  and b.shrcd in (10, 11)
  and b.exchcd in (1, 2, 3)
"""

crsp = db.raw_sql(crsp_sql, date_cols=["date"])
crsp = coerce_numeric(crsp, ["ret", "prc", "shrout", "shrcd", "exchcd"])
crsp = crsp.dropna(subset=["date", "permno", "ret", "prc", "shrout"]).copy()

# CRSP market cap: abs(price) * shares outstanding
# shrout is in thousands, but that's fine for weights
crsp["mkt_cap"] = crsp["prc"].abs() * crsp["shrout"]

# -----------------------------
# 6) Pull CRSP-Compustat link table
# -----------------------------
# These link filters are standard for common link history.
ccm_sql = """
select
    gvkey,
    lpermno as permno,
    linktype,
    linkprim,
    linkdt,
    linkenddt
from crsp.ccmxpf_linktable
where lpermno is not null
  and linktype in ('LU','LC','LS')
  and linkprim in ('P','C')
"""
ccm = db.raw_sql(ccm_sql, date_cols=["linkdt", "linkenddt"])
ccm["linkenddt"] = ccm["linkenddt"].fillna(pd.Timestamp("2100-12-31"))

# -----------------------------
# 7) Pull Compustat company GICS sector
# -----------------------------
# This uses Compustat company-level GICS sector.
# For many student projects this is a practical MVP.
comp_sql = """
select
    gvkey,
    conm,
    gsector
from comp.company
where gsector is not null
"""
comp = db.raw_sql(comp_sql)
comp["gsector"] = comp["gsector"].astype(str).str.strip()

# -----------------------------
# 8) Merge CRSP -> CCM -> Compustat sector
# -----------------------------
# First merge CRSP with link table on permno, then keep valid date ranges.
merged = crsp.merge(ccm, how="left", on="permno")
merged = merged[
    (merged["date"] >= merged["linkdt"]) &
    (merged["date"] <= merged["linkenddt"])
].copy()

# If multiple gvkeys survive for a permno-date, keep first after sorting
merged = merged.sort_values(["permno", "date", "gvkey"]).drop_duplicates(["permno", "date"], keep="first")

# Add GICS sector
merged = merged.merge(comp[["gvkey", "gsector"]], how="left", on="gvkey")
merged["gsector"] = merged["gsector"].astype(str).str.strip()
merged["sector"] = merged["gsector"].map(GICS_MAP)

# Keep only mapped sectors
merged = merged[merged["sector"].notna()].copy()

# -----------------------------
# 9) Build lagged market cap and sector returns
# -----------------------------
merged = merged.sort_values(["permno", "date"]).copy()
merged["lag_mkt_cap"] = merged.groupby("permno")["mkt_cap"].shift(1)

# Drop rows without valid lagged cap or return
merged = merged.dropna(subset=["lag_mkt_cap", "ret"]).copy()
merged = merged[merged["lag_mkt_cap"] > 0].copy()

# Normalize monthly date to month-end just in case
merged["date"] = pd.to_datetime(merged["date"]) + pd.offsets.MonthEnd(0)

# Value-weighted stock weights within each sector-date
sector_totals = merged.groupby(["date", "sector"], as_index=False)["lag_mkt_cap"].sum()
sector_totals = sector_totals.rename(columns={"lag_mkt_cap": "sector_lag_cap"})
merged = merged.merge(sector_totals, on=["date", "sector"], how="left")
merged["weight"] = merged["lag_mkt_cap"] / merged["sector_lag_cap"]

# Sector returns
sector_returns = merged.groupby(["date", "sector"], as_index=False).apply(
    lambda x: pd.Series({
        "sector_ret": np.sum(x["weight"] * x["ret"]),
        "n_stocks": x["permno"].nunique(),
        "sector_lag_cap": x["sector_lag_cap"].iloc[0]
    })
).reset_index(drop=True)

# Create full sector-date grid so ranking works cleanly
all_dates = pd.date_range(sector_returns["date"].min(), sector_returns["date"].max(), freq="M")
all_sectors = list(GICS_MAP.values())
grid = pd.MultiIndex.from_product([all_dates, all_sectors], names=["date", "sector"]).to_frame(index=False)

sector_returns = grid.merge(sector_returns, how="left", on=["date", "sector"]).sort_values(["sector", "date"])

# -----------------------------
# 10) Pull FRED macro data (optional overlay + dashboard context)
# -----------------------------
fred_start = pd.to_datetime(START_DATE) - pd.DateOffset(months=24)

fred_series = {}
for series_code in ["DGS10", "DGS2", "UNRATE", "INDPRO"]:
    try:
        s = web.DataReader(series_code, "fred", fred_start, END_DATE)
        fred_series[series_code] = s
    except Exception as e:
        print(f"Could not pull FRED series {series_code}: {e}")

if len(fred_series) > 0:
    fred = pd.concat(fred_series.values(), axis=1)
    fred.columns = list(fred_series.keys())
    fred = fred.resample("M").last().reset_index().rename(columns={"DATE": "date"})
    fred["date"] = pd.to_datetime(fred["DATE"] if "DATE" in fred.columns else fred["date"]) + pd.offsets.MonthEnd(0)
    fred["yield_spread_10y_2y"] = fred["DGS10"] - fred["DGS2"]
else:
    fred = pd.DataFrame(columns=["date", "yield_spread_10y_2y", "UNRATE", "INDPRO"])

def label_macro_regime(spread):
    if pd.isna(spread):
        return np.nan
    if spread < 0:
        return "Contraction"
    elif spread < 1:
        return "Neutral"
    else:
        return "Expansion"

if not fred.empty:
    fred["macro_regime"] = fred["yield_spread_10y_2y"].apply(label_macro_regime)

# -----------------------------
# 11) Signal engine
# -----------------------------
def build_signals(sector_df, lookback_months=6, skip_months=1, top_n=3, bottom_n=3, use_macro_overlay=False):
    df = sector_df.copy().sort_values(["sector", "date"])

    # Momentum = compounded sector return over lookback window, skipping most recent months
    df["momentum_raw"] = (
        df.groupby("sector")["sector_ret"]
          .transform(lambda x: x.shift(skip_months).rolling(lookback_months).apply(compute_compound_return, raw=False))
    )

    # Cross-sectional ranks
    df["rank"] = df.groupby("date")["momentum_raw"].rank(ascending=False, method="first")
    df["n_sectors"] = df.groupby("date")["sector"].transform("count")

    # z-score version too
    df = zscore_by_date(df, "momentum_raw")
    df = df.rename(columns={"zscore": "momentum_z"})

    # Optional macro overlay
    df["final_score"] = df["momentum_z"]
    df["macro_adjustment"] = 0.0
    df["macro_regime"] = np.nan

    if use_macro_overlay and not fred.empty:
        df = df.merge(fred[["date", "yield_spread_10y_2y", "macro_regime"]], on="date", how="left")

        def macro_boost(row):
            reg = row["macro_regime"]
            sec = row["sector"]
            if pd.isna(reg):
                return 0.0
            if reg == "Expansion" and sec in CYCLICALS:
                return 0.25
            if reg == "Contraction" and sec in DEFENSIVES:
                return 0.25
            return 0.0

        df["macro_adjustment"] = df.apply(macro_boost, axis=1)
        df["final_score"] = df["momentum_z"] + df["macro_adjustment"]

        # rerank final score if overlay is used
        df["rank"] = df.groupby("date")["final_score"].rank(ascending=False, method="first")
    else:
        if "yield_spread_10y_2y" not in df.columns:
            df["yield_spread_10y_2y"] = np.nan

    # Traffic lights
    df["signal"] = df.apply(lambda r: traffic_light_from_rank(r["rank"], int(r["n_sectors"]) if not pd.isna(r["n_sectors"]) else 11, top_n, bottom_n), axis=1)

    # Forward one-month return for backtest
    df["fwd_1m_ret"] = df.groupby("sector")["sector_ret"].shift(-1)

    return df

# -----------------------------
# 12) Backtest engine
# -----------------------------
def run_backtest(signal_df, top_n=3):
    df = signal_df.copy()

    # Use only rows with a valid score and forward return
    valid = df.dropna(subset=["rank", "fwd_1m_ret"]).copy()

    # Strategy: equally weight top N sectors, hold next month
    valid["selected"] = valid["rank"] <= top_n

    port = valid[valid["selected"]].groupby("date", as_index=False).agg(
        strategy_ret=("fwd_1m_ret", "mean")
    )

    # Equal-weight benchmark: average next-month return across all sectors
    bench = valid.groupby("date", as_index=False).agg(
        benchmark_ret=("fwd_1m_ret", "mean")
    )

    # Red basket and green basket diagnostics
    green = valid[valid["signal"] == "Green"].groupby("date", as_index=False).agg(
        green_ret=("fwd_1m_ret", "mean")
    )
    red = valid[valid["signal"] == "Red"].groupby("date", as_index=False).agg(
        red_ret=("fwd_1m_ret", "mean")
    )

    out = port.merge(bench, on="date", how="outer").merge(green, on="date", how="left").merge(red, on="date", how="left")
    out = out.sort_values("date").reset_index(drop=True)

    out["strategy_cum"] = (1 + out["strategy_ret"].fillna(0)).cumprod()
    out["benchmark_cum"] = (1 + out["benchmark_ret"].fillna(0)).cumprod()
    out["green_cum"] = (1 + out["green_ret"].fillna(0)).cumprod()
    out["red_cum"] = (1 + out["red_ret"].fillna(0)).cumprod()
    out["spread_ret"] = out["green_ret"] - out["red_ret"]
    out["spread_cum"] = (1 + out["spread_ret"].fillna(0)).cumprod()

    return out

def summarize_performance(bt):
    summary = pd.DataFrame({
        "Metric": [
            "Annualized Return",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Max Drawdown"
        ],
        "Strategy": [
            annualized_return(bt["strategy_ret"]),
            annualized_vol(bt["strategy_ret"]),
            sharpe_ratio(bt["strategy_ret"]),
            max_drawdown(bt["strategy_ret"])
        ],
        "Benchmark": [
            annualized_return(bt["benchmark_ret"]),
            annualized_vol(bt["benchmark_ret"]),
            sharpe_ratio(bt["benchmark_ret"]),
            max_drawdown(bt["benchmark_ret"])
        ],
        "Green-Red Spread": [
            annualized_return(bt["spread_ret"]),
            annualized_vol(bt["spread_ret"]),
            sharpe_ratio(bt["spread_ret"]),
            max_drawdown(bt["spread_ret"])
        ]
    })
    return summary

# -----------------------------
# 13) Initial build
# -----------------------------
signals = build_signals(
    sector_returns,
    lookback_months=DEFAULT_LOOKBACK_MONTHS,
    skip_months=DEFAULT_SKIP_MONTHS,
    top_n=DEFAULT_TOP_N,
    bottom_n=DEFAULT_BOTTOM_N,
    use_macro_overlay=USE_MACRO_OVERLAY
)

backtest = run_backtest(signals, top_n=DEFAULT_TOP_N)
summary = summarize_performance(backtest)

# -----------------------------
# 14) Dashboard function
# -----------------------------
def render_dashboard(start_date, end_date, lookback_months, skip_months, top_n, bottom_n, use_macro_overlay):
    clear_output(wait=True)

    sig = build_signals(
        sector_returns,
        lookback_months=int(lookback_months),
        skip_months=int(skip_months),
        top_n=int(top_n),
        bottom_n=int(bottom_n),
        use_macro_overlay=bool(use_macro_overlay)
    )

    bt = run_backtest(sig, top_n=int(top_n))

    # Filter display range
    start_date = pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)
    end_date   = pd.to_datetime(end_date) + pd.offsets.MonthEnd(0)

    sig_f = sig[(sig["date"] >= start_date) & (sig["date"] <= end_date)].copy()
    bt_f  = bt[(bt["date"] >= start_date) & (bt["date"] <= end_date)].copy()

    if len(sig_f) == 0 or len(bt_f) == 0:
        print("No data in selected range.")
        return

    current_date = sig_f["date"].max()
    current = sig_f[sig_f["date"] == current_date].sort_values("rank").copy()

    perf = summarize_performance(bt_f)

    # Recompute cumulative in selected window
    bt_f["strategy_cum"]  = (1 + bt_f["strategy_ret"].fillna(0)).cumprod()
    bt_f["benchmark_cum"] = (1 + bt_f["benchmark_ret"].fillna(0)).cumprod()
    bt_f["spread_cum"]    = (1 + bt_f["spread_ret"].fillna(0)).cumprod()

    # ---- Print headers
    print("=" * 78)
    print("SECTOR ROTATION DASHBOARD")
    print("=" * 78)
    print(f"Date range: {start_date.date()} to {end_date.date()}")
    print(f"Lookback months: {lookback_months} | Skip months: {skip_months} | Top N: {top_n} | Bottom N: {bottom_n} | Macro overlay: {use_macro_overlay}")
    print(f"Latest signal date: {current_date.date()}")
    print("=" * 78)

    # ---- Current signal table
    show_cols = ["date", "sector", "momentum_raw", "momentum_z", "macro_adjustment", "final_score", "rank", "signal", "n_stocks"]
    current_show = current[show_cols].copy()
    current_show["momentum_raw"] = current_show["momentum_raw"].round(4)
    current_show["momentum_z"] = current_show["momentum_z"].round(3)
    current_show["macro_adjustment"] = current_show["macro_adjustment"].round(3)
    current_show["final_score"] = current_show["final_score"].round(3)

    print("\nCURRENT MONTH SIGNAL TABLE")
    display(current_show.reset_index(drop=True))

    # ---- Performance table
    print("\nBACKTEST SUMMARY")
    display(perf.style.format({
        "Strategy": "{:.2%}",
        "Benchmark": "{:.2%}",
        "Green-Red Spread": "{:.2%}"
    }, na_rep="NaN"))

    # Fix Sharpe rows to plain numbers
    perf2 = perf.copy()
    for c in ["Strategy", "Benchmark", "Green-Red Spread"]:
        perf2.loc[perf2["Metric"] == "Sharpe Ratio", c] = round(float(perf2.loc[perf2["Metric"] == "Sharpe Ratio", c].iloc[0]), 2) if pd.notna(perf2.loc[perf2["Metric"] == "Sharpe Ratio", c].iloc[0]) else np.nan
    display(perf2)

    # ---- Cumulative return chart
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["strategy_cum"], mode="lines", name="Strategy"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["benchmark_cum"], mode="lines", name="Benchmark"))
    fig1.add_trace(go.Scatter(x=bt_f["date"], y=bt_f["spread_cum"], mode="lines", name="Green-Red Spread"))
    fig1.update_layout(
        title="Cumulative Performance",
        xaxis_title="Date",
        yaxis_title="Growth of $1",
        template="plotly_white",
        width=1000,
        height=500
    )
    fig1.show()

    # ---- Heatmap of traffic lights over time
    signal_to_num = {"Red": -1, "Yellow": 0, "Green": 1}
    hm = sig_f[["date", "sector", "signal"]].copy()
    hm["signal_num"] = hm["signal"].map(signal_to_num)
    heat = hm.pivot(index="sector", columns="date", values="signal_num").reindex(all_sectors)

    fig2 = px.imshow(
        heat,
        aspect="auto",
        title="Traffic Light Heatmap (Green=1, Yellow=0, Red=-1)",
        color_continuous_scale=["red", "yellow", "green"],
        zmin=-1,
        zmax=1
    )
    fig2.update_layout(width=1200, height=500)
    fig2.show()

    # ---- Current momentum bar chart
    fig3 = px.bar(
        current.sort_values("rank"),
        x="sector",
        y="momentum_raw",
        color="signal",
        title=f"Current Sector Momentum ({current_date.date()})",
        hover_data=["rank", "final_score", "n_stocks"]
    )
    fig3.update_layout(template="plotly_white", width=1000, height=500)
    fig3.show()

    # ---- Macro chart
    if not fred.empty:
        fred_f = fred[(fred["date"] >= start_date) & (fred["date"] <= end_date)].copy()
        fig4 = go.Figure()
        fig4.add_trace(go.Scatter(x=fred_f["date"], y=fred_f["yield_spread_10y_2y"], mode="lines", name="10Y-2Y Spread"))
        fig4.update_layout(
            title="FRED Macro Overlay Context: 10Y - 2Y Treasury Spread",
            xaxis_title="Date",
            yaxis_title="Spread",
            template="plotly_white",
            width=1000,
            height=400
        )
        fig4.show()

    # ---- Optional file saves
    if SAVE_CSV_OUTPUTS:
        sig_f.to_csv("sector_signals_filtered.csv", index=False)
        bt_f.to_csv("sector_backtest_filtered.csv", index=False)
        current_show.to_csv("current_signal_table.csv", index=False)
        print("\nCSV files saved to current Colab working directory.")

# -----------------------------
# 15) Widgets
# -----------------------------
available_dates = pd.Series(sorted(sector_returns["date"].dropna().unique()))
min_date = available_dates.min().date()
max_date = available_dates.max().date()

start_picker = widgets.DatePicker(description="Start", value=min_date)
end_picker = widgets.DatePicker(description="End", value=max_date)

lookback_slider = widgets.IntSlider(value=DEFAULT_LOOKBACK_MONTHS, min=3, max=12, step=1, description="Lookback")
skip_slider = widgets.IntSlider(value=DEFAULT_SKIP_MONTHS, min=0, max=2, step=1, description="Skip")
topn_slider = widgets.IntSlider(value=DEFAULT_TOP_N, min=1, max=5, step=1, description="Top N")
bottomn_slider = widgets.IntSlider(value=DEFAULT_BOTTOM_N, min=1, max=5, step=1, description="Bottom N")
macro_toggle = widgets.Checkbox(value=USE_MACRO_OVERLAY, description="Macro overlay")
run_button = widgets.Button(description="Run Dashboard", button_style="success")
output = widgets.Output()

def on_run_clicked(_):
    with output:
        render_dashboard(
            start_date=start_picker.value,
            end_date=end_picker.value,
            lookback_months=lookback_slider.value,
            skip_months=skip_slider.value,
            top_n=topn_slider.value,
            bottom_n=bottomn_slider.value,
            use_macro_overlay=macro_toggle.value
        )

run_button.on_click(on_run_clicked)

controls = widgets.VBox([
    widgets.HBox([start_picker, end_picker]),
    widgets.HBox([lookback_slider, skip_slider]),
    widgets.HBox([topn_slider, bottomn_slider]),
    widgets.HBox([macro_toggle, run_button]),
])

display(controls, output)

# Automatically render once on load
with output:
    render_dashboard(
        start_date=start_picker.value,
        end_date=end_picker.value,
        lookback_months=lookback_slider.value,
        skip_months=skip_slider.value,
        top_n=topn_slider.value,
        bottom_n=bottomn_slider.value,
        use_macro_overlay=macro_toggle.value
    )